In [3]:
import re
import pandas as pd
import nltk
from nltk.corpus import stopwords
from transformers import pipeline

# 1. Configurações iniciais e downloads necessários
nltk.download('stopwords')

# 2. Criação do Dataset Local do seu Projeto (Simulando uma base real de E-commerce)
print("Carregando base de feedbacks do projeto...")
dados_projeto = {
    'comentario_original': [
        "O produto chegou quebrado!!! 😡 Péssimo atendimento, não comprem! Link do meu unboxing: https://youtube.com",
        "Gostei bastante do produto, entrega super rápida dentro do prazo... recomendo @loja_oficial #top",
        "Não funcionou no meu aparelho. Tive que devolver, o suporte demorou 3 dias para responder.",
        "Excelente!!! Nota 10/10. O melhor que já comprei até hoje.",
        "O atendimento foi bom, mas o produto demorou quase um mês para ser entregue na minha casa.",
        "Superou todas as minhas expectativas, material de altíssima qualidade!",
        "Péssimo. Dinheiro jogado fora. O aplicativo trava toda hora e não conecta de jeito nenhum.",
        "O preço é meio alto, mas cumpre o que promete de forma honesta. Valeu a pena.",
        "Não gostei. Veio com a cor errada e a caixa toda amassada. Quero meu reembolso.",
        "Muito prático de usar no dia a dia. Recomendo para todo mundo!"
    ]
}

# Transforma a estrutura acima em um DataFrame limpo do Pandas
df = pd.DataFrame(dados_projeto)


# 3. Definição da Função de Limpeza (Passo 1)
def limpar_texto_portfolio(texto):
    """
    Função para higienizar textos não estruturados.
    Remove links, pontuações abusivas e mantém palavras chaves de negação.
    """
    if not isinstance(texto, str):
        return ""
    
    # Letras minúsculas
    texto = texto.lower()
    
    # Remove Links/URLs
    texto = re.sub(r'https?://\S+|www\.\S+', '', texto)
    
    # Remove arrobas (@) e hashtags (#)
    texto = re.sub(r'@\w+|#\w+', '', texto)
    
    # Remove pontuações e números, preservando acentuação em português
    texto = re.sub(r'[^a-zA-Záéíóúàèìòùâêîôûãõç\s]', '', texto)
    
    # Remove espaços em branco duplos
    texto = " ".join(texto.split())
    
    # Filtro de Stop Words (Palavras vazias)
    palavras_vazias = set(stopwords.words('portuguese'))
    palavras_vazias.discard('não')
    palavras_vazias.discard('nem')
    palavras_vazias.discard('nunca')
    
    palavras_filtradas = [palavra for palavra in texto.split() if palavra not in palavras_vazias]
    
    return " ".join(palavras_filtradas)


# 4. Executando a Limpeza nos Dados
print("Aplicando o pré-processamento de texto...")
df['comentario_limpo'] = df['comentario_original'].apply(limpar_texto_portfolio)


# 5. Executando a Inteligência Artificial (Passo 2)
print("\nCarregando modelo de IA da comunidade... (Aguarde)")
# Usando um modelo multilíngue consolidado e estável para análise de sentimento
classificador_sentimento = pipeline(
    "text-classification", 
    model="cardiffnlp/twitter-xlm-roberta-base-sentiment"
)

# Tradutor interno para o Dashboard ficar elegante
def traduzir_label(label):
    lbl = str(label).lower()
    if 'positive' in lbl: return 'Positivo'
    if 'negative' in lbl: return 'Negativo'
    return 'Neutro'

print("Processando os sentimentos com IA...")
resultados_ia = []
for texto in df['comentario_limpo']:
    predicao = classificador_sentimento(texto)[0]
    resultados_ia.append({
        'sentimento': traduzir_label(predicao['label']),
        'confianca_ia': round(predicao['score'], 4)
    })

# Unifica os resultados ao DataFrame principal do projeto
df_final = pd.concat([df, pd.DataFrame(resultados_ia)], axis=1)


# 6. Exibe o Painel Final de Resultados no seu Terminal
print("\n" + "="*40)
print("--- RESULTADO PROCESSADO PELA IA ---")
print("="*40)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
print(df_final[['comentario_limpo', 'sentimento', 'confianca_ia']])


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\vinus\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Carregando base de feedbacks do projeto...
Aplicando o pré-processamento de texto...

Carregando modelo de IA da comunidade... (Aguarde)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Processando os sentimentos com IA...

--- RESULTADO PROCESSADO PELA IA ---
                                    comentario_limpo sentimento  confianca_ia
0  produto chegou quebrado péssimo atendimento nã...   Negativo        0.9372
1  gostei bastante produto entrega super rápida d...   Positivo        0.8595
2  não funcionou aparelho devolver suporte demoro...   Negativo        0.6530
3                 excelente nota melhor comprei hoje   Positivo        0.8899
4  atendimento bom produto demorou quase mês entr...   Positivo        0.4716
5  superou todas expectativas material altíssima ...   Positivo        0.7688
6  péssimo dinheiro jogado aplicativo trava toda ...   Negativo        0.9141
7  preço meio alto cumpre promete forma honesta v...   Positivo        0.6169
8  não gostei veio cor errada caixa toda amassada...   Negativo        0.9175
9          prático usar dia dia recomendo todo mundo   Positivo        0.6327


In [12]:
# Salva o resultado final em uma planilha CSV pronta para ser consumida
df_final.to_csv("feedbacks_clientes_processados.csv", index=False, encoding="utf-8-sig")
print("\n[SUCESSO] Dados salvos localmente em 'feedbacks_clientes_processados.csv'!")



[SUCESSO] Dados salvos localmente em 'feedbacks_clientes_processados.csv'!
